# Stage97 — split the *existing* private Saturn Track1 in your Drive

This notebook runs under **your own Colab/Google account**. It mounts your private Google Drive, detects the already-owned 621,755,904-byte Track1, validates its **full Stage02 SHA-256**, splits it into five independently readable ZIP files, and copies those plus a manifest to `VH-Saturn/Track1-parts`.

Nothing is uploaded to GitHub, sent to ChatGPT, or published. Do not share this notebook's private runtime or output folder. Run it only if you trust and authorize its code; mounting Drive grants this notebook access to files in that account. The source game is **not** modified or deleted.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # Requires your explicit Google authorization

from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
CHECKOUT = Path('/content/vh-saturn-stage97-code')
if not CHECKOUT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch',
                    'stage97/private-track1-split-zip',
                    'https://github.com/stanmasked/vandal-hearts-saturn.git',
                    str(CHECKOUT)], check=True)
sys.path.insert(0, str(CHECKOUT))
from tools.saturn_private_capture_session import PINNED
from tools.saturn_private_track1_split_zip import split_private_track1, MANIFEST
print('Stage97 code loaded. Original game files have not been changed.')


In [ ]:
# Change only DRIVE_FOLDER if VH-Saturn is not at the root of My Drive.
DRIVE_FOLDER = Path('/content/drive/MyDrive/VH-Saturn')
if not DRIVE_FOLDER.is_dir():
    raise RuntimeError('VH-Saturn folder not found at configured Drive path')
expected_size = PINNED['track1'][0]
originals = [p for p in DRIVE_FOLDER.iterdir()
             if p.is_file() and p.stat().st_size == expected_size]
if len(originals) != 1:
    raise RuntimeError('Expected exactly one Track1 of the Stage02 pinned size')
original_track1 = originals[0]

# Generate all files on Colab's private temporary filesystem FIRST.
LOCAL_PARTS = Path('/content/private-vh-saturn-track1-parts')
LOCAL_PARTS.mkdir(mode=0o700, exist_ok=False)
LOCAL_PARTS.chmod(0o700)
result = split_private_track1(original_track1, LOCAL_PARTS, chunk_bytes=128*1024*1024)
assert result['zip_archives_created'] == 5

# Copy only the five verified ZIPs first; the manifest is copied LAST.
DESTINATION = DRIVE_FOLDER / 'Track1-parts'
if DESTINATION.exists():
    if not DESTINATION.is_dir() or any(DESTINATION.iterdir()):
        raise RuntimeError('Drive output folder exists and is not empty: refusing overwrite')
else:
    DESTINATION.mkdir()

manifest = json.loads((LOCAL_PARTS / MANIFEST).read_text())
def sha256_file(path):
    digest=hashlib.sha256()
    with path.open('rb') as src:
        while block := src.read(1024*1024):
            digest.update(block)
    return digest.hexdigest()

for p in manifest['parts']:
    source = LOCAL_PARTS / p['archive']
    dest = DESTINATION / p['archive']
    with source.open('rb') as incoming, dest.open('xb') as outgoing:
        shutil.copyfileobj(incoming, outgoing, length=1024*1024)
    if dest.stat().st_size != p['archive_bytes'] or sha256_file(dest) != p['archive_sha256']:
        raise RuntimeError('Drive-copy mismatch: do not use incomplete ZIP parts')
    print('Verified ZIP upload', manifest['parts'].index(p)+1, 'of', len(manifest['parts']))

with (LOCAL_PARTS / MANIFEST).open('rb') as src, (DESTINATION / MANIFEST).open('xb') as out:
    shutil.copyfileobj(src, out)
if sha256_file(DESTINATION / MANIFEST) != sha256_file(LOCAL_PARTS / MANIFEST):
    raise RuntimeError('Drive-copy manifest mismatch')
print('SUCCESS: five independent ZIP parts and private JSON manifest copied to Google Drive.')
print('Original Track1 unchanged. First-battle/game boot NOT verified.')
